# Aquaculture Model Training

This notebook demonstrates the training workflow for the aquaculture ML framework using actual competition data.


## 1. Setup and Configuration


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import os
import random
from pathlib import Path
import sys

# Add the parent directory to the system path to import local modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import our custom modules
from aquaculture.feature_engineering import AquacultureFeatureEngineer
from aquaculture.feature_selection import FeatureSelector, select_temporal_features, select_metadata_features
from src.config import TrainingConfig
from src.trainer import Trainer

# For reproducibility
np.random.seed(42)
random.seed(42)

# Define where your data files live.
# Adjust this path if your data are located elsewhere.
DATA_DIR = Path("../data")      # relative to the notebook's working directory
# Verify the directory exists
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR.resolve()}")

# Define the path to the experiment directory where models and results will be saved
EXPERIMENT_DIR = Path("../experiments")  # relative to the notebook's working directory
# Create the experiment directory if it doesn't exist
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Data Loading and Preparation


In [ ]:
# Load training data from CSV file
print("Loading training data...")
train_df = pd.read_csv(DATA_DIR / 'Train.csv')
print(f"Training data shape: {train_df.shape}")
print(f"Training data columns: {list(train_df.columns)}")

# Prepare data for training
print("Preparing data for training...")
# The target column is 'label' in the training data
# Feature columns are all columns except ID and label
feature_cols = [col for col in train_df.columns if col not in ['ID', 'label']]
X = train_df[feature_cols].values

# Get target variable - binary classification: 0 (no pond) or 1 (pond)
print("Extracting target variable from 'label' column...")
y = train_df['label'].values
print(f"Target variable shape: {y.shape}")
print(f"Target distribution: {np.bincount(y.astype(int)) if len(y) > 0 else 'empty'}")


## 3. Model Training and Evaluation


In [ ]:
# Create a configuration object
config = TrainingConfig()                     # <-- instantiate the config

# Select model type (e.g., 'lightgbm', 'catboost', or 'xgboost')
config.model_type = 'lightgbm'                # <-- choose your model type

# Select enginerring features to use
config.feature_engineering_config.include_optical = True
config.feature_engineering_config.include_sar = True
config.feature_engineering_config.include_temporal_statistics = True
config.feature_engineering_config.include_cross_sensor_features = True
config.feature_engineering_config.include_metadata = True

# Feature selection configuration (disabled by default, meaning all features are used)
# To enable feature selection, set feature_selection_enabled = True
# To customize feature selection, modify feature_selection_method and feature_selection_kwargs
config.feature_selection_enabled = False      # Set to True to enable feature selection
config.feature_selection_method = 'groups'    # Method to use for feature selection
config.feature_selection_kwargs = {}          # Keyword arguments for the selection method

# Enable SHAP
config.compute_shap = True

# Set experiment directory in config
config.experiment_dir = EXPERIMENT_DIR

# Initialize trainer
trainer = Trainer(config)

# Feature selection information
print("Feature selection configuration:")
print(f"  Enabled: {config.feature_selection_enabled}")
print(f"  Method: {config.feature_selection_method}")
print(f"  Kwargs: {config.feature_selection_kwargs}")
if config.feature_selection_enabled:
    print("  Feature selection is ENABLED - only selected features will be used for training")
else:
    print("  Feature selection is DISABLED - all features will be used for training (default behavior)")

# Examples of how to configure feature selection (uncomment to use):
#
# Example 1: Use only temporal and metadata features
# config.feature_selection_enabled = True
# config.feature_selection_method = 'groups'
# config.feature_selection_kwargs = {'groups': ['temporal', 'metadata']}
#
# Example 2: Use only features with specific patterns (e.g., mean and std)
# config.feature_selection_enabled = True
# config.feature_selection_method = 'patterns'
# config.feature_selection_kwargs = {'patterns': ['_mean$', '_std$']}
#
# Example 3: Combine selections - temporal + metadata, but exclude specific features
# config.feature_selection_enabled = True
# config.feature_selection_method = 'combine'
# config.feature_selection_kwargs = {
#     'include': {'method': 'groups', 'groups': ['temporal', 'metadata']},
#     'exclude': {'method': 'names', 'names': ['green_01', 'nir_01']}
# }

In [ ]:
# Train models
print("Training models...")
trainer.fit(X, y)

# Save the trainer (including feature selector) for later use in inference
print("Saving trainer for later use...")
trainer.save()
print("Trainer saved successfully!")

In [ ]:
# Evaluate training performance with observation simulation (matches training conditions)
print("\\n=== Training set evaluation (with observation stimulation) ===")
train_preds = trainer.predict(X, training=True)
train_probas = trainer.predict_proba(X, training=True)[:, 1]

from src.metrics import calculate_metrics, competition_score
metrics = calculate_metrics(y, train_probas)
print("Metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

comp = competition_score(y, train_probas)
print(f"  competition_score: {comp:.4f}")

# OPTIONAL: Feature Selection Evaluation
# Uncomment the following to see how selected features perform
#
# Example: Evaluate using only temporal features
# print("\\n--- Evaluation with temporal features only ---")
# temporal_selector = select_temporal_features(trainer.feature_engineer)
# X_temporal_df = temporal_selector.transform(X, training=True)
# X_temporal = X_temporal_df.values
# 
# temporal_preds = trainer.predict(X_temporal, training=True)
# temporal_probas = trainer.predict_proba(X_temporal, training=True)[:, 1]
# 
# temporal_metrics = calculate_metrics(y, temporal_probas)
# print("Temporal features metrics:")
# for k, v in temporal_metrics.items():
#     print(f"  {k}: {v:.4f}")
# 
# temporal_comp = competition_score(y, temporal_probas)
# print(f"  competition_score: {temporal_comp:.4f}")
# 
# print(f"\\nFeature count comparison:")
# print(f"  All features: {trainer._last_X_features.shape[1]}")
# print(f"  Temporal features: {X_temporal_df.shape[1]}")